# SentiScope — Model Training Notebook

This notebook walks through the full ML pipeline:
1. Data loading & exploration
2. Text preprocessing
3. Feature extraction (TF-IDF)
4. Model training (Logistic Regression)
5. Evaluation & metrics

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✓')

## 1. Load Dataset

Using built-in sample data. Replace `df` with your own CSV for better results (e.g., IMDb reviews, Amazon reviews).

In [ ]:
import sys
sys.path.append('..')
from model.sentiment_model import SAMPLE_DATA

df = pd.DataFrame(SAMPLE_DATA, columns=['text', 'label'])
print(f'Dataset size: {len(df)}')
df.head()

In [ ]:
# Label distribution
fig, ax = plt.subplots(figsize=(6,3))
df['label'].value_counts().plot(kind='bar', color=['#22c55e','#ef4444','#f59e0b'], ax=ax)
ax.set_title('Label Distribution')
ax.set_xlabel('Sentiment')
ax.set_ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 2. Text Preprocessing

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z0-9\s!?.,\']', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(clean_text)
df[['text','clean_text']].head(4)

In [ ]:
# Text length stats
df['word_count'] = df['clean_text'].apply(lambda x: len(x.split()))
print(df.groupby('label')['word_count'].describe())

## 3. Train/Test Split & Pipeline

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label'],
    test_size=0.2, random_state=42, stratify=df['label']
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=5000,
        sublinear_tf=True,
    )),
    ('clf', LogisticRegression(
        max_iter=1000, C=1.0,
        multi_class='multinomial', solver='lbfgs'
    ))
])

pipeline.fit(X_train, y_train)
print('Model trained ✓')

## 4. Evaluation

In [ ]:
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=['positive','neutral','negative'])
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['positive','neutral','negative'],
            yticklabels=['positive','neutral','negative'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 5. Top TF-IDF Features

In [ ]:
vocab = pipeline.named_steps['tfidf'].get_feature_names_out()
coefs = pipeline.named_steps['clf'].coef_
classes = pipeline.named_steps['clf'].classes_

for i, cls in enumerate(classes):
    top_idx = np.argsort(coefs[i])[-10:][::-1]
    print(f'\nTop words → {cls.upper()}')
    print([vocab[j] for j in top_idx])

## 6. Try Your Own Text

In [ ]:
test_texts = [
    "This movie was absolutely phenomenal!",
    "I regret buying this, complete waste.",
    "It's okay, nothing to rave about.",
]

for t in test_texts:
    pred  = pipeline.predict([clean_text(t)])[0]
    proba = pipeline.predict_proba([clean_text(t)])[0]
    conf  = max(proba) * 100
    print(f'{pred.upper():10s} ({conf:.1f}%)  →  "{t}"')